# IKG Lineage Workbench

End-to-end helper for:
- refreshing the lineage dependency table/Excel directly from `sandbox_prj_smart_insights.ikg_table_lineage_metadata_auto_refresh`
- stitching ordered SQL sources from `sandbox_prj_smart_insights.ikg_table_lineage_auto_refresh_temp`
- producing the `_new`, `_modified`, and `_modified_profile` SQL scripts ready for execution.

In [ ]:
import logging

from ikg_lineage_sql_stitcher import (
    GitLabSQLFetcher,
    build_dependency_artifacts,
    derive_forced_table_tokens,
    enforce_profile_table_suffix,
    fetch_lineage_entries,
    fetch_profile_scripts,
    parse_insight_values,
    prepare_temp_table_plan,
    stitch_sql,
    render_stitched_text,
    write_output_file,
    write_modified_file,
    build_profile_only_text,
    write_profile_only_file,
    run_sql_script,
    preview_final_table,
    display_table_counts,
    find_tables_for_suffix,
    EXCLUDE_FOLDER,
    TARGET_SCHEMA,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
print("Lineage workbench helpers ready.")

In [ ]:
import datetime
import os
from pathlib import Path
import getpass

import gitlab
import psycopg2
from psycopg2 import sql

print("Environment ready.")

In [ ]:
INSIGHT_RAW = input("Enter insight_type (comma-separated if needed): ").strip()
if not INSIGHT_RAW:
    raise ValueError("insight_type is required.")

INSIGHT_VALUES = parse_insight_values(INSIGHT_RAW)
if not INSIGHT_VALUES:
    raise ValueError("Please provide at least one insight_type.")

PROFILE_DATE = input("Enter profile date (YYYYMMDD): ").strip()
if not PROFILE_DATE:
    raise ValueError("profile date is required.")

ACC_MHH_N = input("Enter acc_mhh_n (optional, press Enter to skip): ").strip()
SANDBOX_SCHEMA = input(
    f"Enter sandbox schema for temp tables [{TARGET_SCHEMA}]: "
).strip() or TARGET_SCHEMA

DB_PASSWORD = getpass.getpass("Enter Password for DB User: ")
DB_CONFIG = {
    "host": "greenplum-rdsp.zur.swissbank.com",
    "port": "5432",
    "dbname": "gprdsp",
    "user": "ds_rdsp_dev",
    "password": DB_PASSWORD,
}
print("Captured credentials for Greenplum.")

In [ ]:
RUN_TIMESTAMP = datetime.datetime.utcnow()
LINEAGE_ROWS, PROFILE_TABLES, EXCEL_PATH, ALIAS_LOOKUP = build_dependency_artifacts(
    INSIGHT_VALUES,
    DB_CONFIG,
    run_timestamp=RUN_TIMESTAMP,
)
FORCED_TABLES = derive_forced_table_tokens(PROFILE_TABLES)
TEMP_PLAN = prepare_temp_table_plan(DB_CONFIG, SANDBOX_SCHEMA, ACC_MHH_N)
TEMP_TABLE_MAP = TEMP_PLAN.table_map

if PROFILE_TABLES:
    print(f"Dependency builder returned {len(LINEAGE_ROWS)} rows and Excel at {EXCEL_PATH}.")
else:
    print("No profile tables identified; downstream steps will exit early.")
print(f"Temp table map size: {len(TEMP_TABLE_MAP)}")
print(f"Alias lookup entries: {len(ALIAS_LOOKUP)}")

In [ ]:
print(f"Forced table tokens ({len(FORCED_TABLES)}): {sorted(FORCED_TABLES)}")
print(f"Temp table map entries: {list(TEMP_TABLE_MAP.items())[:3]}")

In [ ]:
with psycopg2.connect(**DB_CONFIG) as conn:
    entries = fetch_lineage_entries(conn)

print(f"Retrieved {len(entries)} lineage entries.")
print(f"Profile tables to append: {PROFILE_TABLES}")
entries[:5]

In [ ]:
PRIVATE_TOKEN = getpass.getpass("Enter your private token: ")
exclude_folders = [folder for folder in EXCLUDE_FOLDER.split() if folder]
fetcher = GitLabSQLFetcher(private_token=PRIVATE_TOKEN, exclude_folders=exclude_folders)
print("GitLab fetcher initialized.")

In [ ]:
stitched_sql = stitch_sql(entries, fetcher, ALIAS_LOOKUP)
profile_scripts = fetch_profile_scripts(fetcher, PROFILE_TABLES)
if profile_scripts:
    processed_profiles = []
    for (path, content), table_name in zip(profile_scripts, PROFILE_TABLES):
        processed_profiles.append((path, enforce_profile_table_suffix(content, table_name)))
    stitched_sql.extend(processed_profiles)
else:
    if PROFILE_TABLES:
        print("Warning: No profile scripts fetched; profile-only execution will be skipped.")
stitched_text = render_stitched_text(stitched_sql)
print(f"Stitched {len(stitched_sql)} SQL files (including {len(profile_scripts)} profile scripts).")

In [ ]:
primary_profile_table = PROFILE_TABLES[-1] if PROFILE_TABLES else None
output_new = write_output_file(INSIGHT_RAW, stitched_text, primary_profile_table)
output_modified, modified_text = write_modified_file(
    INSIGHT_RAW,
    stitched_text,
    PROFILE_DATE,
    FORCED_TABLES,
    TEMP_TABLE_MAP,
    SANDBOX_SCHEMA,
    primary_profile_table,
)
print(f"Created files: {output_new} (original), {output_modified} (modified)")

created_tables = sorted(find_tables_for_suffix(modified_text))
run_sql_script(modified_text, DB_CONFIG)
display_table_counts(created_tables, DB_CONFIG)
preview_final_table(DB_CONFIG, PROFILE_TABLES)

if profile_scripts:
    profile_only_text = build_profile_only_text(
        profile_scripts[-1][1],
        PROFILE_DATE,
        FORCED_TABLES,
        TEMP_TABLE_MAP,
        SANDBOX_SCHEMA,
    )
    profile_only_path = write_profile_only_file(
        INSIGHT_RAW,
        profile_only_text,
        primary_profile_table,
    )
    run_sql_script(profile_only_text, DB_CONFIG)
    preview_final_table(DB_CONFIG, PROFILE_TABLES)
    print(f"Profile-only script executed from {profile_only_path}.")
else:
    print("No profile scripts available; skipping execution and preview.")